# Notebook 67 — Policy Learning

**Telemetry specifies policy.**

Notebook 61 telemetry rows become an auditable routing policy; Notebook 71 evaluates the policy offline. Outputs go to `figures/`, `data/`, and `artifacts/`.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, zipfile
FIG_DIR=Path("figures"); DATA_DIR=Path("data"); ARTIFACT_DIR=Path("artifacts")
for d in [FIG_DIR,DATA_DIR,ARTIFACT_DIR]: d.mkdir(exist_ok=True)
rng=np.random.default_rng(67); n=900
domain=rng.choice(["math","code","qa","safety","long_context"],n)
confidence=np.clip(rng.beta(5,3,n),.02,.99); entropy=np.clip(1.7-1.35*confidence+rng.normal(0,.18,n),.05,2.1); margin=np.clip(.7*confidence-.2*entropy+rng.normal(.06,.1,n),0,.95)
risk=np.clip(rng.beta(2,5,n)+.35*(domain=="safety"),0,1); latency=np.clip(rng.normal(650,220,n),80,1400); pressure=1-(latency-latency.min())/(latency.max()-latency.min()); disagree=np.clip(.15+.5*(entropy>1.05)+.25*(risk>.65)+rng.normal(0,.13,n),0,1)
actions=[]
for c,e,m,r,b,v in zip(confidence,entropy,margin,risk,pressure,disagree):
    if r>.72 or v>.72: a="verify"
    elif b>.82 and c>.62 and m>.24: a="stop"
    elif c<.42 and b>.35: a="deepen"
    elif c<.38 and b<=.35: a="fallback"
    elif e>1.1 and m<.22: a="deepen"
    else: a="continue"
    actions.append(a)
reward=np.clip(.55+.2*confidence+.12*margin-.14*entropy-.07*pressure+rng.normal(0,.05,n),0,1)
df=pd.DataFrame({"domain":domain,"confidence":confidence,"entropy":entropy,"margin":margin,"risk_score":risk,"latency_budget_ms":latency,"budget_pressure":pressure,"verifier_disagreement":disagree,"learned_action":actions,"reward":reward})
df.to_csv(DATA_DIR/"notebook67_policy_telemetry.csv",index=False)
fig,ax=plt.subplots(); df["learned_action"].value_counts().reindex(["continue","deepen","verify","stop","fallback"]).plot(kind="bar",ax=ax); ax.set_title("Notebook 67 — policy action distribution"); fig.tight_layout(); fig.savefig(FIG_DIR/"67_action_distribution.png",dpi=180); plt.show()
fig,ax=plt.subplots();
for a,g in df.groupby("learned_action"): ax.scatter(g.entropy,g.confidence,s=12,alpha=.35,label=a)
ax.set_title("Telemetry state space: confidence vs entropy"); ax.set_xlabel("Entropy"); ax.set_ylabel("Confidence"); ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR/"67_confidence_entropy_state_space.png",dpi=180); plt.show()
# simple policy surface
cg=np.linspace(.05,.98,80); eg=np.linspace(.05,2.05,80); Z=[]
order=["continue","deepen","fallback","stop","verify"]; amap={a:i for i,a in enumerate(order)}
for e in eg:
    row=[]
    for c in cg:
        m=max(0,min(.95,.7*c-.2*e+.06)); r=.35; b=.5; v=.15+.5*(e>1.05)
        if r>.72 or v>.72: a="verify"
        elif b>.82 and c>.62 and m>.24: a="stop"
        elif c<.42 and b>.35: a="deepen"
        elif c<.38 and b<=.35: a="fallback"
        elif e>1.1 and m<.22: a="deepen"
        else: a="continue"
        row.append(amap[a])
    Z.append(row)
fig,ax=plt.subplots(); im=ax.imshow(Z,origin="lower",aspect="auto",extent=[cg.min(),cg.max(),eg.min(),eg.max()]); ax.set_title("Policy surface: confidence × entropy"); ax.set_xlabel("Confidence"); ax.set_ylabel("Entropy"); cb=fig.colorbar(im,ax=ax,ticks=list(amap.values())); cb.ax.set_yticklabels(order); fig.tight_layout(); fig.savefig(FIG_DIR/"67_policy_surface_confidence_entropy.png",dpi=180); plt.show()
fig,ax=plt.subplots(); pd.crosstab(pd.cut(df.budget_pressure,6),df.learned_action,normalize="index").plot(kind="bar",stacked=True,ax=ax); ax.set_title("Policy action mix as latency pressure rises"); ax.set_ylabel("Action share"); fig.tight_layout(); fig.savefig(FIG_DIR/"67_budget_aware_routing.png",dpi=180); plt.show()
card={"notebook":"67_policy_learning","statement":"Telemetry specifies policy.","previous":"61_telemetry_dataset","next":"71_offline_evaluation","actions":order,"figures":sorted([p.name for p in FIG_DIR.glob("*.png")])}
json.dump(card,open(ARTIFACT_DIR/"notebook67_policy_card.json","w"),indent=2)
zip_path=Path("notebook67_policy_learning_outputs.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR,DATA_DIR,ARTIFACT_DIR]:
        for p in folder.rglob("*"): z.write(p,p)
    z.write("67_policy_learning.ipynb")
print(zip_path)

## Notebook 71 handoff

Use `data/notebook67_policy_telemetry.csv` and `artifacts/notebook67_policy_card.json` as the offline-evaluation bridge.